In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp /content/drive/MyDrive/cotton_baseline/cotton_clean.zip /content/
!unzip -q /content/cotton_clean.zip -d /
!cat /content/cotton_clean/data.yaml
!ls /content/cotton_clean/train/images | wc -l

In [ ]:
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
!pip install -r requirements.txt

In [ ]:
!sed -i 's/default_act = nn.SiLU()/default_act = nn.ReLU()/' models/common.py
!grep -n "default_act" models/common.py

In [ ]:
!python train.py --img 640 --batch 16 --epochs 100 \
  --data /content/cotton_clean/data.yaml \
  --weights yolov5n.pt --name cotton_relu --cache

In [ ]:
!mkdir -p /content/drive/MyDrive/cotton_relu
!cp -r runs/train/cotton_relu /content/drive/MyDrive/cotton_relu/

In [ ]:
!python val.py --weights runs/train/cotton_relu/weights/best.pt \
  --data /content/cotton_clean/data.yaml --img 640 --task test \
  --max-det 3000 --conf 0.001

In [ ]:
import torch
ckpt = torch.load('runs/train/cotton_relu/weights/best.pt', map_location='cpu', weights_only=False)
torch.save(ckpt['model'].float().state_dict(), 'best_relu_state_dict.pt')
!cp best_relu_state_dict.pt /content/drive/MyDrive/cotton_relu/
print('ok')

In [ ]:
!python export.py --weights runs/train/cotton_relu/weights/best.pt --include onnx --img 640 --opset 12
!cp runs/train/cotton_relu/weights/best.onnx /content/drive/MyDrive/cotton_relu/best_relu.onnx